In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# --- 1. 저장된 데이터 불러오기 ---
try:
    raw_data = np.load('raw_channel_data.npy')
    print(f"✅ 데이터 로드 성공! Shape: {raw_data.shape}") # (42000, 624) 나와야 함
except FileNotFoundError:
    print("❌ 데이터 파일이 없습니다. 이전 단계에서 저장을 먼저 해주세요.")

# --- 2. 파라미터 설정 (논문 3.4장 기준) ---
WINDOW_SIZE = 60        # 과거 60개 (Input)
PREDICTION_HORIZON = 14 # 미래 14개 뒤 (Target)
SUB_CARRIERS = 91       # 논문: 중심 주파수 기준 91개만 사용 (계산 효율성)
CENTER_IDX = 624 // 2   # 중심 인덱스 (312)
SNR_DB = 20.0           # 학습에 사용할 잡음 레벨 (일단 20dB로 시작)

# --- 3. 전처리 함수 정의 ---
def preprocess_pipeline(data, snr_db):
    print(f"\n>>> 전처리 시작 (SNR: {snr_db}dB)...")
    
    # [Step 1] 관심 부반송파(91개) 자르기
    # 312번을 중심으로 좌우 45개씩 -> 총 91개
    start = CENTER_IDX - (SUB_CARRIERS // 2)
    end = CENTER_IDX + (SUB_CARRIERS // 2) + 1
    data_cropped = data[:, start:end]
    print(f"   - 부반송파 크롭 완료: {data.shape} -> {data_cropped.shape}")
    
    # [Step 2] 잡음(Noise) 추가
    # 논문 핵심: 입력(X)은 잡음이 있고, 정답(Y)은 잡음이 없는 깨끗한 데이터여야 함
    
    # 신호 전력 계산
    sig_power = np.mean(np.abs(data_cropped)**2)
    # 목표 잡음 전력 계산 (SNR 공식 역산)
    noise_power = sig_power / (10**(snr_db/10))
    
    # 복소수 잡음 생성 (Real, Imag 각각 생성)
    noise = (np.random.randn(*data_cropped.shape) + 1j * np.random.randn(*data_cropped.shape)) \
            * np.sqrt(noise_power/2)
            
    noisy_input = data_cropped + noise  # 입력용 (더러운 데이터)
    clean_target = data_cropped         # 정답용 (깨끗한 데이터)
    
    # [Step 3] 슬라이딩 윈도우 (Sliding Window)
    # X: t ~ t+59 (60개)
    # Y: t+59+14 (14번째 뒤의 미래 값)
    
    num_samples = len(data_cropped) - WINDOW_SIZE - PREDICTION_HORIZON
    X_list = []
    Y_list = []
    
    print(f"   - 슬라이딩 윈도우 생성 중 (샘플 수: {num_samples})...")
    # for문은 느리지만 이해하기 쉬움 (4만개 정도는 금방 됩니다)
    for i in range(num_samples):
        # 입력: 잡음 섞인 과거 60개
        X_list.append(noisy_input[i : i+WINDOW_SIZE])
        # 정답: 미래의 깨끗한 값 1개
        Y_list.append(clean_target[i + WINDOW_SIZE + PREDICTION_HORIZON - 1])
        
    X = np.array(X_list) # [Samples, 60, 91]
    Y = np.array(Y_list) # [Samples, 91]
    
    # [Step 4] 실수/허수 분리
    print("   - 실수/허수 분리 중...")
    X_real = np.real(X)
    X_imag = np.imag(X)
    Y_real = np.real(Y)
    Y_imag = np.imag(Y)
    
    # [Step 5] 정규화 (MinMax Scaling: 0~1 사이로)
    # 주의: 나중에 복원하려면 min, max 값을 저장해둬야 함
    def get_minmax(arr):
        return np.min(arr), np.max(arr)
    
    # 전체 데이터 기준 Min/Max 계산 (간단한 방식)
    min_xr, max_xr = get_minmax(X_real)
    min_xi, max_xi = get_minmax(X_imag)
    min_yr, max_yr = get_minmax(Y_real)
    min_yi, max_yi = get_minmax(Y_imag)
    
    # 스케일링 적용 함수
    def scale(arr, min_v, max_v):
        return (arr - min_v) / (max_v - min_v)
    
    X_real = scale(X_real, min_xr, max_xr)
    X_imag = scale(X_imag, min_xi, max_xi)
    Y_real = scale(Y_real, min_yr, max_yr)
    Y_imag = scale(Y_imag, min_yi, max_yi)
    
    # 복원용 정보 저장 (Dictionary)
    scalers = {
        'xr': (min_xr, max_xr), 'xi': (min_xi, max_xi),
        'yr': (min_yr, max_yr), 'yi': (min_yi, max_yi)
    }
    
    print("✅ 전처리 완료!")
    return (X_real, X_imag), (Y_real, Y_imag), scalers

# --- 4. 실행 ---
(X_r, X_i), (Y_r, Y_i), scalers = preprocess_pipeline(raw_data, SNR_DB)

# --- 5. 데이터 확인 (Shape) ---
print(f"\n최종 데이터 확인:")
print(f"입력(Real) Shape: {X_r.shape}") # (Samples, 60, 91)
print(f"정답(Real) Shape: {Y_r.shape}") # (Samples, 91)

# --- 6. Train / Test 분리 (논문대로 2/3 : 1/3 비율) ---
# 논문에서는 시계열을 뚝 잘라서 앞부분을 Train, 뒷부분을 Test로 씀 (Shuffle False)
split_idx = int(len(X_r) * (2/3))

X_r_train, X_r_test = X_r[:split_idx], X_r[split_idx:]
Y_r_train, Y_r_test = Y_r[:split_idx], Y_r[split_idx:]

X_i_train, X_i_test = X_i[:split_idx], X_i[split_idx:]
Y_i_train, Y_i_test = Y_i[:split_idx], Y_i[split_idx:]

print(f"\n[데이터셋 분할]")
print(f"Train Set: {X_r_train.shape}")
print(f"Test Set : {X_r_test.shape}")

# 저장해두기 (선택사항, 메모리 넉넉하면 변수 그대로 사용)

✅ 데이터 로드 성공! Shape: (42000, 624)

>>> 전처리 시작 (SNR: 20.0dB)...
   - 부반송파 크롭 완료: (42000, 624) -> (42000, 91)
   - 슬라이딩 윈도우 생성 중 (샘플 수: 41926)...
   - 실수/허수 분리 중...
✅ 전처리 완료!

최종 데이터 확인:
입력(Real) Shape: (41926, 60, 91)
정답(Real) Shape: (41926, 91)

[데이터셋 분할]
Train Set: (27950, 60, 91)
Test Set : (13976, 60, 91)


In [2]:
import pickle

print(">>> 전처리된 데이터 저장을 시작합니다...")

# 1. 학습용 데이터(Numpy 배열) 저장 (.npz 파일)
np.savez('processed_dataset.npz',
         X_r_train=X_r_train, X_r_test=X_r_test,
         Y_r_train=Y_r_train, Y_r_test=Y_r_test,
         X_i_train=X_i_train, X_i_test=X_i_test,
         Y_i_train=Y_i_train, Y_i_test=Y_i_test)

# 2. 스케일러 정보(최소/최대값) 저장 (.pkl 파일) -> 나중에 복원할 때 필수!
with open('scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print("✅ 저장 완료!")
print("   - 데이터 파일: processed_dataset.npz")
print("   - 스케일러 파일: scalers.pkl")

>>> 전처리된 데이터 저장을 시작합니다...
✅ 저장 완료!
   - 데이터 파일: processed_dataset.npz
   - 스케일러 파일: scalers.pkl


In [3]:
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# --- 1. 데이터 로드 ---
try:
    raw_data = np.load('raw_channel_data.npy')
    print(f"✅ 데이터 로드 성공! Shape: {raw_data.shape}")
except FileNotFoundError:
    print("❌ 데이터 파일이 없습니다.")

# --- 2. 파라미터 ---
WINDOW_SIZE = 60
PREDICTION_HORIZON = 14
SUB_CARRIERS = 91
CENTER_IDX = 624 // 2
SNR_DB = 20.0

# --- 3. 개선된 전처리 함수 (StandardScaler 사용) ---
def preprocess_pipeline_improved(data, snr_db):
    print(f"\n>>> [개선된 전처리] 시작 (SNR: {snr_db}dB)...")
    
    # [Step 1] 크롭
    start = CENTER_IDX - (SUB_CARRIERS // 2)
    end = CENTER_IDX + (SUB_CARRIERS // 2) + 1
    data_cropped = data[:, start:end]
    
    # [Step 2] 잡음 추가
    sig_power = np.mean(np.abs(data_cropped)**2)
    noise_power = sig_power / (10**(snr_db/10))
    noise = (np.random.randn(*data_cropped.shape) + 1j * np.random.randn(*data_cropped.shape)) * np.sqrt(noise_power/2)
    
    noisy_input = data_cropped + noise
    clean_target = data_cropped
    
    # [Step 3] 윈도우 생성
    num_samples = len(data_cropped) - WINDOW_SIZE - PREDICTION_HORIZON
    X_list, Y_list = [], []
    
    for i in range(num_samples):
        X_list.append(noisy_input[i : i+WINDOW_SIZE])
        Y_list.append(clean_target[i + WINDOW_SIZE + PREDICTION_HORIZON - 1])
        
    X = np.array(X_list) # (N, 60, 91)
    Y = np.array(Y_list) # (N, 91)
    
    # [Step 4] 실수/허수 분리
    # Shape을 (N, Feature) 형태로 펴서 스케일링 해야 함
    # X는 (N, 60, 91)이므로 스케일링을 위해 잠시 형태를 바꿈
    N = X.shape[0]
    
    X_real = np.real(X).reshape(N, -1) # (N, 60*91)
    X_imag = np.imag(X).reshape(N, -1)
    Y_real = np.real(Y) # (N, 91)
    Y_imag = np.imag(Y)
    
    # [Step 5] StandardScaler 적용 (중요!)
    # 평균을 0으로, 분산을 1로 맞춰줍니다.
    scaler_xr = StandardScaler()
    scaler_xi = StandardScaler()
    scaler_yr = StandardScaler()
    scaler_yi = StandardScaler()
    
    print("   - StandardScaler 학습 및 적용 중...")
    X_real = scaler_xr.fit_transform(X_real)
    X_imag = scaler_xi.fit_transform(X_imag)
    Y_real = scaler_yr.fit_transform(Y_real)
    Y_imag = scaler_yi.fit_transform(Y_imag)
    
    # 다시 원래 모양으로 복구 (LSTM 입력용)
    X_real = X_real.reshape(N, WINDOW_SIZE, SUB_CARRIERS)
    X_imag = X_imag.reshape(N, WINDOW_SIZE, SUB_CARRIERS)
    
    # 스케일러 객체 저장 (나중에 복원할 때 필요)
    scalers = {
        'xr': scaler_xr, 'xi': scaler_xi,
        'yr': scaler_yr, 'yi': scaler_yi
    }
    
    return (X_real, X_imag), (Y_real, Y_imag), scalers

# --- 4. 실행 및 저장 ---
(X_r, X_i), (Y_r, Y_i), scalers = preprocess_pipeline_improved(raw_data, SNR_DB)

# 데이터 분할
split_idx = int(len(X_r) * (2/3))
X_r_train, X_r_test = X_r[:split_idx], X_r[split_idx:]
Y_r_train, Y_r_test = Y_r[:split_idx], Y_r[split_idx:]
X_i_train, X_i_test = X_i[:split_idx], X_i[split_idx:]
Y_i_train, Y_i_test = Y_i[:split_idx], Y_i[split_idx:]

# 저장
print(">>> 저장 중...")
np.savez('processed_dataset.npz',
         X_r_train=X_r_train, X_r_test=X_r_test,
         Y_r_train=Y_r_train, Y_r_test=Y_r_test,
         X_i_train=X_i_train, X_i_test=X_i_test,
         Y_i_train=Y_i_train, Y_i_test=Y_i_test)

with open('scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print("✅ 개선된 전처리 및 저장 완료!")

✅ 데이터 로드 성공! Shape: (42000, 624)

>>> [개선된 전처리] 시작 (SNR: 20.0dB)...
   - StandardScaler 학습 및 적용 중...
>>> 저장 중...
✅ 개선된 전처리 및 저장 완료!
